# 02 - Exploratory Data Analysis & Statistical Testing
**Platform:** Bangladesh E-Commerce Customer Intelligence Platform  
**Focus:** Revenue Dynamics, Customer Spending Skewness, Seasonality, & Hypothesis Testing

---
## Business Questions Answered
1. What is the empirical spending distribution of Bangladeshi e-commerce consumers?
2. Does the revenue distribution follow the Pareto principle (80/20 rule)?
3. How do major cultural and religious retail peaks (Eid-ul-Fitr, Eid-ul-Adha, 11.11 Mega Sale) impact revenue?
4. Is there a statistically significant spending disparity between Dhaka and non-Dhaka consumers?

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)

from src.analytics.eda import load_curated_data, compute_spending_summary, compute_pareto_distribution, perform_statistical_hypothesis_tests

df_cust, df_prod, df_ord, df_items = load_curated_data()
print("Data loaded successfully.")

## 1. Spending Distribution & Summary Statistics

In [ ]:
spend_stats = compute_spending_summary(df_ord)
for k, v in spend_stats.items():
    print(f"{k}: {v:,.2f}" if isinstance(v, float) else f"{k}: {v}")

## 2. Pareto Revenue Concentration (80/20 Rule)

In [ ]:
pareto_df = compute_pareto_distribution(df_ord)

top_10_pct_rev = pareto_df[pareto_df['cumulative_customer_pct'] <= 10]['cumulative_revenue_pct'].max()
top_20_pct_rev = pareto_df[pareto_df['cumulative_customer_pct'] <= 20]['cumulative_revenue_pct'].max()

print(f"Top 10% of customers generate: {top_10_pct_rev:.2f}% of total revenue")
print(f"Top 20% of customers generate: {top_20_pct_rev:.2f}% of total revenue")

plt.figure(figsize=(10, 5))
plt.plot(pareto_df['cumulative_customer_pct'], pareto_df['cumulative_revenue_pct'], color='#1f77b4', lw=2.5, label='Lorenz Revenue Curve')
plt.plot([0, 100], [0, 100], color='gray', linestyle='--', label='Line of Perfect Equality')
plt.title('Customer Revenue Concentration (Lorenz Curve)', fontsize=13, fontweight='bold')
plt.xlabel('Cumulative % of Customers', fontsize=11)
plt.ylabel('Cumulative % of Revenue', fontsize=11)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Monthly Revenue Trajectory & Seasonality

In [ ]:
delivered = df_ord[df_ord['order_status'] == 'Delivered'].copy()
delivered['month'] = delivered['order_date'].dt.to_period('M').astype(str)
monthly_rev = delivered.groupby('month')['total_amount'].sum().reset_index()

plt.figure(figsize=(12, 5))
plt.plot(monthly_rev['month'], monthly_rev['total_amount'] / 1e6, marker='o', color='#2ca02c', lw=2)
plt.title('Monthly Realized Revenue (Million BDT)', fontsize=13, fontweight='bold')
plt.xticks(rotation=45)
plt.ylabel('Million BDT')
plt.grid(True, alpha=0.3)
plt.show()

## 4. Formal Statistical Hypothesis Testing
- **Mann-Whitney U Test:** Dhaka vs. Outside Dhaka customer order values.
- **Kruskal-Wallis Test:** Order spend variations across marketing acquisition channels.
- **Chi-Square Test:** Payment method preference vs. customer type.

In [ ]:
test_results = perform_statistical_hypothesis_tests(df_cust, df_ord)
for test_name, res in test_results.items():
    print(f"=== {test_name.upper()} ===")
    for k, v in res.items():
        print(f"  {k}: {v}")